In [1]:
import numpy as np
import xcdat as xc
import scipy
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt 
from cdo import *   # python version
import scipy.stats as stats
import os

sys.path.append('../../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *
cdo = Cdo()

import shutil
import tempfile
from pathlib import Path

In [7]:
# Cell 3 — Helpers for time chunks and CDO processing
## COPIED FROM processing.ipynb

def realization_number(member_name):
    """Numerical ordering: r1, r2, ..., r10 rather than alphabetical order."""
    return int(re.match(r"r(\d+)", member_name).group(1))

def cdo_input_stream(paths):
    """
    Wrap each variable stream in CDO argument-group brackets.

    This lets CDO distinguish:
      merge(mergetime(rsdt chunks),
            mergetime(rlut chunks),
            mergetime(rsut chunks))
    """
    if len(paths) == 1:
        return ["[", str(paths[0]), "]"]

    return [
        "[",
        "-mergetime",
        *map(str, paths),
        "]",
    ]

def month_number(yyyymm):
    """Convert YYYYMM to a monotonically increasing month number."""
    year = int(yyyymm[:4])
    month = int(yyyymm[4:6])
    return year * 12 + month - 1


def member_paths(row, variable):
    """
    Return a variable's source files in chronological order.

    A normal Maria record has one path.
    A catalog 'ambiguous' record contains semicolon-separated, sequential
    time chunks. These are safe to use when they form one continuous series.
    """
    path_column = f"{variable}_path"
    path_value = row.get(path_column)

    if pd.isna(path_value) or not isinstance(path_value, str):
        raise ValueError(f"{path_column} is missing")

    relative_paths = [
        path.strip()
        for path in path_value.split(";")
        if path.strip()
    ]

    if not relative_paths:
        raise ValueError(f"{path_column} contains no files")

    paths_with_periods = []
    for relative_path in relative_paths:
        match = re.search(r"_(\d{6})-(\d{6})\.nc$", relative_path)
        if not match:
            raise ValueError(f"Could not read date range from {relative_path}")

        start, end = match.groups()
        paths_with_periods.append(
            (month_number(start), month_number(end), maria_root / relative_path.lstrip("./"))
        )

    paths_with_periods.sort()

    # Confirm that adjacent source files are consecutive monthly chunks,
    # rather than duplicate versions or competing grids.
    for (_, previous_end, _), (next_start, _, _) in zip(
        paths_with_periods[:-1],
        paths_with_periods[1:],
    ):
        if next_start != previous_end + 1:
            raise ValueError(
                f"{row['model']} {row['experiment']} {row['member']} {variable}: "
                "source chunks are not continuous in time."
            )

    # Every source collection must cover the desired common output period.
    if paths_with_periods[0][0] > month_number("187001"):
        raise ValueError(f"{variable} starts after 1870-01 for {row['member']}")
    if paths_with_periods[-1][1] < month_number("201412"):
        raise ValueError(f"{variable} ends before 2014-12 for {row['member']}")

    paths = [path for _, _, path in paths_with_periods]

    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing Maria source file(s):\n" + "\n".join(missing))

    return paths


# def create_member_n(row, temporary_n_file):
#     """
#     Merge any sequential time chunks, calculate N, clip to 1870–2014,
#     and conservatively regrid to the low-resolution template.
#     """
#     rsdt_paths = member_paths(row, "rsdt")
#     rlut_paths = member_paths(row, "rlut")
#     rsut_paths = member_paths(row, "rsut")

#     def grouped_stream(paths):
#         # CDO needs [ ... ] to distinguish the three variable streams when
#         # one or more of them uses the multiple-input mergetime operator.
#         if len(paths) == 1:
#             return ["[", str(paths[0]), "]"]

#         return [
#             "[",
#             "-mergetime",
#             *map(str, paths),
#             "]",
#         ]

#     command = [
#         "cdo",
#         "-L",
#         "-O",
#         "-f", "nc4c",
#         "-z", "zip_4",
#         f"remapcon,{lowres_template}",
#         f"-seldate,{start_date},{end_date}",
#         "-expr,N=rsdt-rlut-rsut",
#         "-merge",
#         *grouped_stream(rsdt_paths),
#         *grouped_stream(rlut_paths),
#         *grouped_stream(rsut_paths),
#         str(temporary_n_file),
#     ]

#     print(f"    CDO: {row['member']}")

#     result = subprocess.run(command, text=True, capture_output=True)

#     if result.returncode != 0:
#         print("\nCDO command:")
#         print(" ".join(command))
#         print("\nCDO stderr:")
#         print(result.stderr)
#         raise RuntimeError(
#             f"CDO failed for {row['model']} / "
#             f"{row['experiment']} / {row['member']}"
#         )


def create_member_n(row, temporary_n_file):
    """
    Merge sequential time chunks for rsdt, rlut, and rsut,
    calculate N = rsdt - rlut - rsut,
    clip to 1870–2014,
    and conservatively regrid to the low-resolution template.
    """
    rsdt_paths = member_paths(row, "rsdt")
    rlut_paths = member_paths(row, "rlut")
    rsut_paths = member_paths(row, "rsut")

    # Temporary files for the three merged variables
    tmp_dir = Path(tempfile.mkdtemp(prefix="cdo_inputs_"))

    try:
        rsdt_merged = tmp_dir / "rsdt.nc"
        rlut_merged = tmp_dir / "rlut.nc"
        rsut_merged = tmp_dir / "rsut.nc"
        merged = tmp_dir / "merged.nc"

        def mergetime(paths, output):
            command = [
                "cdo",
                "-L",
                "-O",
                "mergetime",
                *map(str, paths),
                str(output),
            ]

            result = subprocess.run(
                command,
                text=True,
                capture_output=True,
            )

            if result.returncode != 0:
                print("\nCDO command:")
                print(" ".join(command))
                print("\nCDO stderr:")
                print(result.stderr)
                raise RuntimeError(
                    f"CDO mergetime failed for {row['model']} / "
                    f"{row['experiment']} / {row['member']}"
                )

        # Merge each variable's time chunks
        mergetime(rsdt_paths, rsdt_merged)
        mergetime(rlut_paths, rlut_merged)
        mergetime(rsut_paths, rsut_merged)

        # Put the three variables into one file
        command = [
            "cdo",
            "-L",
            "-O",
            "merge",
            str(rsdt_merged),
            str(rlut_merged),
            str(rsut_merged),
            str(merged),
        ]

        result = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )

        if result.returncode != 0:
            print("\nCDO command:")
            print(" ".join(command))
            print("\nCDO stderr:")
            print(result.stderr)
            raise RuntimeError(
                f"CDO merge failed for {row['model']} / "
                f"{row['experiment']} / {row['member']}"
            )

        # Calculate N, select dates, and remap
        command = [
            "cdo",
            "-L",
            "-O",
            "-f", "nc4c",
            "-z", "zip_4",
            f"remapcon,{lowres_template}",
            f"-seldate,{start_date},{end_date}",
            "-expr,N=rsdt-rlut-rsut",
            str(merged),
            str(temporary_n_file),
        ]

        print(f"    CDO: {row['member']}")

        result = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )

        if result.returncode != 0:
            print("\nCDO command:")
            print(" ".join(command))
            print("\nCDO stderr:")
            print(result.stderr)
            raise RuntimeError(
                f"CDO failed for {row['model']} / "
                f"{row['experiment']} / {row['member']}"
            )

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

In [8]:
from pathlib import Path
import pandas as pd
import re
catalog_path = Path("./maria_catalog/maria_catalog.csv")
# catalog_path = Path("amip_clean/maria_catalog.csv")
maria_catalog_df = pd.read_csv(catalog_path)

# Maria's archive root.
maria_root = Path("/rugenstein-archive/mariarug/model_output/MMLEA")

historical_catalog = maria_catalog_df.loc[
    maria_catalog_df["experiment"].eq("historical")
].copy()

In [9]:
# Summarize Maria historical members with complete monthly coverage
# for tas, rsdt, rlut, and rsut over 1870-01 through 2014-12.

required_variables = ("tas", "rsdt", "rlut", "rsut")

# Use the same catalog DataFrame name as in the existing notebook.
# If it is not already loaded, uncomment and adjust:
# maria_catalog = pd.read_csv(maria_catalog_csv)



available_members = []
rejected_members = []

for _, row in historical_catalog.iterrows():
    try:
        paths_by_variable = {
            variable: member_paths(row, variable)
            for variable in required_variables
        }

        # Guard against a helper implementation that returns an empty list
        # instead of raising an exception.
        missing = [
            variable
            for variable, paths in paths_by_variable.items()
            if not paths
        ]
        if missing:
            raise ValueError(f"missing files for: {', '.join(missing)}")

        available_members.append(
            {
                "experiment": row["experiment"],
                "model": row["model"],
                "member": row["member"],
            }
        )

    except (ValueError, FileNotFoundError, OSError, KeyError) as exc:
        rejected_members.append(
            {
                "model": row.get("model"),
                "member": row.get("member"),
                "reason": str(exc),
            }
        )

available_members = (
    pd.DataFrame(available_members)
    .drop_duplicates(["experiment", "model", "member"])
)

historical_summary = (
    available_members
    .groupby(["experiment", "model"], as_index=False)
    .agg(
        n_members=("member", "nunique"),
        members=("member", lambda values: sorted(
            set(values),
            key=lambda value: int(
                re.search(r"r(\d+)", str(value)).group(1)
            )
        )),
    )
    .sort_values(["experiment", "model"])
)

print(
    "Maria members with tas, rsdt, rlut, and rsut covering "
    "1870-01 through 2014-12:"
)

for row in historical_summary.itertuples(index=False):
    print(
        f"  {row.experiment:<16}"
        f"{row.model:<22}"
        f"{row.n_members:>2} member(s)"
    )

Maria members with tas, rsdt, rlut, and rsut covering 1870-01 through 2014-12:
  historical      CNRM-CM6-1            29 member(s)
  historical      CanESM5               25 member(s)
  historical      E3SM-2-0              20 member(s)
  historical      EC-Earth3             23 member(s)
  historical      GFDL-CM4               1 member(s)
  historical      GISS-E2-1-G           19 member(s)
  historical      HadGEM3-GC31-LL       55 member(s)
  historical      IPSL-CM6A-LR          33 member(s)
  historical      MIROC6                50 member(s)
  historical      NorESM2-LM             3 member(s)


In [10]:
## Now move on to the missing ones

In [11]:
## CESM2
from pathlib import Path
from collections import defaultdict
import re
import pandas as pd

cesm_historical_root = Path(
    "/rugenstein-archive/mariarug/model_output/MMLEA/CESM2/historical"
)

cesm_variables = ("TREFHT", "FLUT", "FSNTOA")
target_start = "187001"
target_end = "201412"


def month_number(yyyymm):
    """Convert YYYYMM to a monotonically increasing month number."""
    value = str(yyyymm)
    year = int(value[:4])
    month = int(value[4:6])

    if not 1 <= month <= 12:
        raise ValueError(f"Invalid YYYYMM value: {value}")

    return year * 12 + month - 1


def ranges_cover_window(ranges, start_yyyymm, end_yyyymm):
    """
    Return True when one or more possibly overlapping file ranges provide
    continuous monthly coverage of the requested window.
    """
    target_first = month_number(start_yyyymm)
    target_last = month_number(end_yyyymm)
    cursor = target_first

    numeric_ranges = sorted(
        (month_number(start), month_number(end))
        for start, end, _ in ranges
    )

    for first, last in numeric_ranges:
        if last < cursor:
            continue

        if first > cursor:
            return False

        cursor = max(cursor, last + 1)

        if cursor > target_last:
            return True

    return False


def cesm_member_id(case_name):
    """Extract a native CESM-LENS2 member identifier."""
    match = re.search(r"(LE2-\d{4}\.\d{3})", case_name)
    return match.group(1) if match else case_name


def cesm_member_sort_key(member):
    match = re.fullmatch(r"LE2-(\d{4})\.(\d{3})", member)

    if match:
        return int(match.group(1)), int(match.group(2))

    return float("inf"), str(member)

In [12]:
variable_pattern = "|".join(map(re.escape, cesm_variables))

filename_pattern = re.compile(
    rf"^(?P<case>.+)"
    rf"\.cam\.h0\."
    rf"(?P<variable>{variable_pattern})"
    rf"\.(?P<start>\d{{6}})-(?P<end>\d{{6}})"
    rf"\.nc(?:4)?$"
)

# inventory[case][variable] = [(start, end, path), ...]
inventory = defaultdict(lambda: defaultdict(list))

for path in cesm_historical_root.rglob("*.nc*"):
    match = filename_pattern.match(path.name)

    if match is None:
        continue

    case = match.group("case")

    # Matches BHIST, BHISTsmbb, and similar CESM historical case names.
    if "BHIST" not in case:
        continue

    inventory[case][match.group("variable")].append(
        (
            match.group("start"),
            match.group("end"),
            path,
        )
    )

if not inventory:
    raise RuntimeError(
        f"No CESM BHIST files matching TREFHT, FLUT, or FSNTOA "
        f"were found under {cesm_historical_root}"
    )

rows = []

for case, variable_files in inventory.items():
    row = {
        "case": case,
        "member": cesm_member_id(case),
    }

    for variable in cesm_variables:
        ranges = variable_files.get(variable, [])

        row[f"{variable}_files"] = len(ranges)
        row[f"{variable}_covers_1870_2014"] = (
            ranges_cover_window(ranges, target_start, target_end)
            if ranges
            else False
        )

    row["complete"] = all(
        row[f"{variable}_covers_1870_2014"]
        for variable in cesm_variables
    )

    rows.append(row)

cesm_historical_inventory = pd.DataFrame(rows)

complete_cesm_members = sorted(
    cesm_historical_inventory.loc[
        cesm_historical_inventory["complete"], "member"
    ].unique(),
    key=cesm_member_sort_key,
)

print(
    "CESM2 BHIST members with TREFHT, FLUT, and FSNTOA "
    "covering 1870-01 through 2014-12:"
)
print(f"  {len(complete_cesm_members)} member(s)")

for member in complete_cesm_members:
    print(f"    {member}")

# Useful for diagnosing members that did not qualify.
display(
    cesm_historical_inventory.sort_values(
        "member",
        key=lambda column: column.map(cesm_member_sort_key),
    ).reset_index(drop=True)
)

CESM2 BHIST members with TREFHT, FLUT, and FSNTOA covering 1870-01 through 2014-12:
  50 member(s)
    LE2-1011.001
    LE2-1031.002
    LE2-1051.003
    LE2-1071.004
    LE2-1091.005
    LE2-1111.006
    LE2-1131.007
    LE2-1151.008
    LE2-1171.009
    LE2-1191.010
    LE2-1231.011
    LE2-1231.012
    LE2-1231.013
    LE2-1231.014
    LE2-1231.015
    LE2-1231.016
    LE2-1231.017
    LE2-1231.018
    LE2-1231.019
    LE2-1231.020
    LE2-1251.011
    LE2-1251.012
    LE2-1251.013
    LE2-1251.014
    LE2-1251.015
    LE2-1251.016
    LE2-1251.017
    LE2-1251.018
    LE2-1251.019
    LE2-1251.020
    LE2-1281.011
    LE2-1281.012
    LE2-1281.013
    LE2-1281.014
    LE2-1281.015
    LE2-1281.016
    LE2-1281.017
    LE2-1281.018
    LE2-1281.019
    LE2-1281.020
    LE2-1301.011
    LE2-1301.012
    LE2-1301.013
    LE2-1301.014
    LE2-1301.015
    LE2-1301.016
    LE2-1301.017
    LE2-1301.018
    LE2-1301.019
    LE2-1301.020


,case,member,TREFHT_files,TREFHT_covers_1870_2014,FLUT_files,FLUT_covers_1870_2014,FSNTOA_files,FSNTOA_covers_1870_2014,complete
0,b.e21.BHISTsmbb.f09_g17.LE2-1011.001,LE2-1011.001,17,True,17,True,17,True,True
1,b.e21.BHISTsmbb.f09_g17.LE2-1031.002,LE2-1031.002,17,True,17,True,17,True,True
2,b.e21.BHISTsmbb.f09_g17.LE2-1051.003,LE2-1051.003,17,True,17,True,17,True,True
3,b.e21.BHISTsmbb.f09_g17.LE2-1071.004,LE2-1071.004,17,True,17,True,17,True,True
4,b.e21.BHISTsmbb.f09_g17.LE2-1091.005,LE2-1091.005,17,True,17,True,17,True,True
5,b.e21.BHISTsmbb.f09_g17.LE2-1111.006,LE2-1111.006,17,True,17,True,17,True,True
6,b.e21.BHISTsmbb.f09_g17.LE2-1131.007,LE2-1131.007,17,True,17,True,17,True,True
7,b.e21.BHISTsmbb.f09_g17.LE2-1151.008,LE2-1151.008,17,True,17,True,17,True,True
8,b.e21.BHISTsmbb.f09_g17.LE2-1171.009,LE2-1171.009,17,True,17,True,17,True,True
9,b.e21.BHISTsmbb.f09_g17.LE2-1191.010,LE2-1191.010,17,True,17,True,17,True,True


In [14]:
from pathlib import Path
import re
import pandas as pd

eth_manifest_path = Path(
    "remote_structure_manifest_maria_ETH.txt"
)

required_variables = frozenset({
    "tas",
    "rlut",
    "rsdt",
    "rsut",
})

eth_filename_pattern = re.compile(
    r"^\./"
    r"(?P<variable>tas|rlut|rsdt|rsut)_latlon_mon/"
    r"(?P=variable)_mon_"
    r"(?P<model>.+)_historical_"
    r"(?P<member>r\d+i\d+p\d+(?:f\d+)?)_"
    r"(?P<grid>g025|native)\.nc$"
)


def cmip_member_sort_key(member):
    """Sort r1 before r2 before r10, while respecting i/p/f labels."""
    match = re.fullmatch(
        r"r(\d+)i(\d+)p(\d+)(?:f(\d+))?",
        str(member),
    )

    if match is None:
        return (float("inf"), str(member))

    realization, initialization, physics, forcing = match.groups()

    return (
        int(realization),
        int(initialization),
        int(physics),
        int(forcing) if forcing is not None else 0,
    )


records = []

with eth_manifest_path.open() as manifest:
    for line in manifest:
        match = eth_filename_pattern.match(line.strip())

        if match is not None:
            records.append(match.groupdict())

eth_files = pd.DataFrame(records).drop_duplicates()

if eth_files.empty:
    raise RuntimeError(
        f"No matching monthly historical files found in "
        f"{eth_manifest_path}"
    )

# Prevent native and g025 copies from being counted as separate members.
member_variables = (
    eth_files
    .groupby(["model", "member"], as_index=False)
    .agg(
        variables=("variable", lambda values: frozenset(values)),
        grids=("grid", lambda values: sorted(set(values))),
    )
)

eth_historical_members = member_variables.loc[
    member_variables["variables"].map(
        lambda variables: required_variables.issubset(variables)
    )
].copy()

eth_historical_summary = (
    eth_historical_members
    .groupby("model", as_index=False)
    .agg(
        n_members=("member", "nunique"),
        members=(
            "member",
            lambda values: sorted(
                set(values),
                key=cmip_member_sort_key,
            ),
        ),
    )
    .sort_values("model")
    .reset_index(drop=True)
)

print(
    "ETH monthly historical members with "
    "tas, rsdt, rlut, and rsut:"
)

for row in eth_historical_summary.itertuples(index=False):
    print(
        f"  {'historical':<16}"
        f"{row.model:<26}"
        f"{row.n_members:>3} member(s)"
    )

print(
    f"\nTotal: {len(eth_historical_summary)} model(s), "
    f"{len(eth_historical_members)} model-member combination(s)"
)

ETH monthly historical members with tas, rsdt, rlut, and rsut:
  historical      ACCESS-CM2                  3 member(s)
  historical      ACCESS-ESM1-5              30 member(s)
  historical      AWI-CM-1-1-MR               5 member(s)
  historical      AWI-ESM-1-1-LR              1 member(s)
  historical      BCC-CSM2-MR                 3 member(s)
  historical      BCC-ESM1                    3 member(s)
  historical      CAMS-CSM1-0                 3 member(s)
  historical      CAS-ESM2-0                  4 member(s)
  historical      CESM2                      11 member(s)
  historical      CESM2-FV2                   3 member(s)
  historical      CESM2-WACCM                 3 member(s)
  historical      CESM2-WACCM-FV2             3 member(s)
  historical      CIESM                       3 member(s)
  historical      CMCC-CM2-HR4                1 member(s)
  historical      CMCC-CM2-SR5                1 member(s)
  historical      CMCC-ESM2                   1 member(s)
  histori

In [15]:
xc.open_dataset('../data/rlut_mon_CanESM5_historical_r1i1p1f1_g025.nc')

<xarray.Dataset> Size: 164MB
Dimensions:   (time: 1980, lat: 72, lon: 144, bnds: 2)
Coordinates:
  * time      (time) object 16kB 1850-01-16 12:00:00 ... 2014-12-16 12:00:00
  * lat       (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * lon       (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
Dimensions without coordinates: bnds
Data variables:
    rlut      (time, lat, lon) float64 164MB ...
    lon_bnds  (lon, bnds) float64 2kB 0.0 2.5 2.5 5.0 ... 357.5 357.5 360.0
    lat_bnds  (lat, bnds) float64 1kB -90.0 -87.5 -87.5 -85.0 ... 87.5 87.5 90.0
Attributes: (12/58)
    CDI:                         Climate Data Interface version 1.9.6 (http:/...
    history:                     Thu Dec 19 08:53:45 2019: cdo -O -b F64 -rem...
    source:                      CanESM5 (2019): \naerosol: interactive\natmo...
    institution:                 Canadian Centre for Climate Modelling and An...
    Conventions:                 CF-1.7 CMIP-6.2
    CCCma_model_hash:            3dedf95315d603326fde4f5340dc0519d80d10c0
    ...                          ...
    license:                     CMIP6 model data produced by The Government ...
    cmor_version:                3.4.0
    cmip6-ng:                    \ncontact = cmip6-archive@env.ethz.ch\ndescr...
    original_file_names:         /net/atmos/data/cmip6/historical/Amon/rlut/C...
    original_file_hash_codes:    9672682edc364e3b299dffbea6aa4f30aed3e26e8dfa...
    CDO:                         Climate Data Operators version 1.9.6 (http:/...

In [16]:
eth_historical_summary

,model,n_members,members
0,ACCESS-CM2,3,"[r1i1p1f1, r2i1p1f1, r3i1p1f1]"
1,ACCESS-ESM1-5,30,"[r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p1f1, r5i1p..."
2,AWI-CM-1-1-MR,5,"[r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p1f1, r5i1p..."
3,AWI-ESM-1-1-LR,1,[r1i1p1f1]
4,BCC-CSM2-MR,3,"[r1i1p1f1, r2i1p1f1, r3i1p1f1]"
5,BCC-ESM1,3,"[r1i1p1f1, r2i1p1f1, r3i1p1f1]"
6,CAMS-CSM1-0,3,"[r1i1p1f1, r1i1p1f2, r2i1p1f1]"
7,CAS-ESM2-0,4,"[r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p1f1]"
8,CESM2,11,"[r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p1f1, r5i1p..."
9,CESM2-FV2,3,"[r1i1p1f1, r2i1p1f1, r3i1p1f1]"
